# Counting touching objects workflow
---
*Introduction to Image Analysis Workshop*

*Stefania Marcotti (stefania.marcotti@crick.ac.uk)*

*Seeded watershed for segmenting touching objects*

*CC-BY-SA-4.0 license: creativecommons.org/licenses/by-sa/4.0/*

*Adapted from Tom Slater (slatert2@cardiff.ac.uk)*

*[Intro to building image analysis pipelines for electron microscopy data with Python](https://github.com/RMS-DAIM/introduction-to-image-analysis/blob/main/Scripts/Jupyter/electron_microscopy_analysis.ipynb)*

---

# Introduction

In this notebook, we will explore how to approach the segmentation of touching objects using a seeded watershed.

Watershedding is commonly used to separate touching or overlapping objects in microscopy and materials imaging. 

By the end of this notebook, you should be able to:

- Segment touching objects using a watershed algorithm
- Generate markers for watershed segmentation

## Importing libraries

We first import the libraries required for the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy import ndimage as ndi

from skimage import feature
from skimage import measure
from skimage import filters
from skimage import segmentation
from skimage import io

import pandas as pd

## Watershed segmentation for electron microscopy data
We'll first load a high-angle annular dark field (HAADF) scanning transmission electron microscope (STEM) image of SiO2 nanoparticles. The vast majority of electron microscopy images are single-channel data, i.e., they are already grayscale images with only two dimensions (y, x).

In [ ]:
im = io.imread("../../Data/demos/SiO2_HAADF_Image.tiff")

print('Image dimensions:', im.shape)
print('Image type:', type(im))

In [ ]:
# display image
fig, ax = plt.subplots(figsize=(6,4))
ax.imshow(im, cmap='gray')
ax.set_title('SiO2 nanoparticles')
ax.axis('off')
plt.tight_layout()

## Thresholding the image

We use grey-level thresholding to obtain a binary mask, similarly to what we've done in previous sessions.

In [ ]:
threshold = filters.threshold_otsu(im)

im_thresh = im > threshold

fig, ax = plt.subplots(figsize=(6,4))
ax.imshow(im_thresh, cmap='gray')
ax.set_title('Binary image')
ax.axis('off')
plt.tight_layout()

If we apply the same labelling approach that we've used previously, we are not able to accurately label each nanoparticle due to the connectedness of each label. We could try exploring different thresholding algorithms, but it will be difficult to separate touching nanoparticles.

In [ ]:
# label objects and visualise the result
labels = measure.label(im_thresh)

fig, ax = plt.subplots(figsize=(6,4))
ax.imshow(labels, cmap='jet')
ax.set_title('Label image')
ax.axis('off')
plt.tight_layout()

In [ ]:
# count the objects - find the maximum integer assigned to a label!
print('There are', labels.max(), 'objects in the image')

## Picking out seeds for the watershed algorithm

The watershed algorithm often uses a [distance transform](https://neubias.github.io/training-resources/distance_transform/index.html) to locate the centres of objects. Distance transforms quantify how a structure of interest is away from object boundaries or other structures.

In [ ]:
# calculate distance transform of the binary mask
distance = ndi.distance_transform_edt(im_thresh)

fig, ax = plt.subplots(1,2,figsize=(8,4))

ax[0].imshow(distance, cmap='magma')
ax[0].set_title('Distance transform')
ax[0].axis('off')

ax[1].imshow(distance[10:40, 10:40], cmap='magma')
ax[1].set_title('Distance transform (zoom)')
ax[1].axis('off')

plt.tight_layout()

The local maxima of the distance transform can then be used to seed the [watershed algorithm](https://neubias.github.io/training-resources/watershed/index.html). Please note that seeding is not mandatory when performing watershedding; however, it might ensure better accuracy. To calculate the local maxima, we can use a `skimage.feature` function called [`peak_local_max`](https://scikit-image.org/docs/stable/auto_examples/segmentation/plot_peak_local_max.html). This function takes as inputs the distance transform, a parameter called `min_distance` which decides the minimum separation allowed between peaks, and the label image.

In [ ]:
# locate local maxima by first calculating their coordinates
coords = feature.peak_local_max(distance, min_distance=3, labels=labels)

fig, ax = plt.subplots(figsize=(6,4))
ax.imshow(im, cmap='gray')
ax.plot(coords[:, 1], coords[:, 0], 'y.')
ax.axis('off')
ax.set_title('Peak local max')

plt.tight_layout()

<div style="background-color:#abd9e9; border-radius: 5px; padding: 10pt">
<strong>Task</strong>
What would change if the <code>min_distance</code> parameter were set to a higher or lower value? Test it below </div>

In [ ]:
# Test a different `min_distance`
coords_test = [...]

# display the raw image with the peak local maxima superimposed
fig, ax = plt.subplots(figsize=(6,4))
# add your code to visualise your image here

## Applying watershed segmentation

Another `skimage` function, this time in the `segmentation` library, can perform watershed segmentation ([`skimage.segmentation.watershed`](https://scikit-image.org/docs/stable/api/skimage.segmentation.html#skimage.segmentation.watershed)). The function takes three inputs: an image, the seed markers, and the binary mask of the raw image.

The first input expects a data array where the lowest value points are labelled first. This works out to be the inverse of the distance transform, so that the centres of each object are now highlighted as local minima.

The seed markers for the second input need to be organised as an array with the same dimensions as the input image. At the moment, we have the markers' coordinates saved in an array; therefore, the first step is to create an empty image with the correct size and add the markers as labels in the correct locations.

In [ ]:
# create an empty image with the same shape as the input image
mask = np.zeros(im.shape, dtype=bool)

# add the markers as True pixels in the coordinates stored in coords (this is a binary mask!)
mask[tuple(coords.T)] = True

# label each True pixel (this is a label mask!)
markers = measure.label(mask)

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(10,4))

ax[0].imshow(im, cmap='gray')
ax[0].set_title('Original image')
ax[0].axis('off')

ax[1].imshow(mask, cmap='gray')
ax[1].set_title('Seed markers (binary)')
ax[1].axis('off')

ax[2].imshow(markers, cmap='jet')
ax[2].set_title('Seed markers (label)')
ax[2].axis('off')

plt.tight_layout()

We can now apply the watershed algorithm. The segmentation obtained with the watershed function is equivalent to that returned by `measure.label` on the binary mask of the original image, although in this case, we have many more objects!

In [ ]:
labels_watershed = segmentation.watershed(-distance, markers, mask=im_thresh)

fig, ax = plt.subplots(1, 2, figsize=(8,4))

ax[0].imshow(-distance, cmap='magma')
ax[0].set_title('Inverted distance transform')
ax[0].axis('off')

ax[1].imshow(labels_watershed, cmap='jet')
ax[1].set_title('Watershed segmentation')
ax[1].axis('off')

plt.tight_layout()

In [ ]:
# count the objects - find the maximum integer assigned to a label!
print('There are', labels_watershed.max(), 'objects in the image')

## Measuring segmented objects

We can extract measurements from the labelled regions in exactly the same manner as previously.

In [ ]:
# measure properties
props = measure.regionprops_table(labels, im, properties=['area', 'centroid', 'eccentricity'])
props_df = pd.DataFrame(props)

props_df.head()

In [ ]:
# display some results
fig, axs = plt.subplots(1, 2, figsize=(4,3))

axs[0].boxplot(props_df['area'])
axs[0].set_title('Particle area (px)')

axs[1].boxplot(props_df['eccentricity'])
axs[1].set_title('Particle eccentricity')

plt.tight_layout()